In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

c:\Users\KIIT\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset
df = pd.read_csv("../data/cleaned_transcripts.csv")

print("Dataset Loaded ✅")
df.head()

Dataset Loaded ✅


,video_id,title,datetime,transcript
0,ldkfMvE36FI,How to handle being thrown into an existing co...,2026-03-08 12:43:26+00:00,theres going to be a need for people can like ...
1,jie_039IekA,Why you shouldnt chase people to get what you ...,2026-03-07 13:32:30+00:00,I dont do a ton of followup Like you know if i...
2,mrRfPVm9nAY,Learn the basics of Data Structures in 60 seco...,2026-03-06 13:18:50+00:00,Lets learn the basics of data structures in le...
3,hP931079TMw,There are 2 kinds of devs One of them is screw...,2026-03-06 11:01:15+00:00,Welcome back to the Free Code Camp podcast Im ...
4,tVskbekONlw,Learn MLOps with MLflow and Databricks Full Co...,2026-03-05 14:21:18+00:00,This course is an end toend guide to mastering...


In [3]:
# Load best model
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

print(f"Model Loaded: {model_name} ✅")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3766.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded: all-MiniLM-L6-v2 ✅


In [4]:
# -----------------------------
# Step 4: Generate Title Embeddings
# -----------------------------

# Extract titles
titles = df['title'].astype(str).tolist()

print("Generating title embeddings... ⏳")

# Generate embeddings
title_embeddings = model.encode(titles, show_progress_bar=True)

print("Title embeddings generated successfully ✅")

# Check shape
print("Shape of title embeddings:", title_embeddings.shape)

Generating title embeddings... ⏳


Batches: 100%|██████████| 8/8 [00:00<00:00,  9.53it/s]

Title embeddings generated successfully ✅
Shape of title embeddings: (245, 384)


In [5]:
# -----------------------------
# Step 5: Generate Transcript Embeddings
# -----------------------------

# Extract transcripts
transcripts = df['transcript'].astype(str).tolist()

print("Generating transcript embeddings... ⏳")

# Generate embeddings
transcript_embeddings = model.encode(transcripts, show_progress_bar=True)

print("Transcript embeddings generated successfully ✅")

# Check shape
print("Shape of transcript embeddings:", transcript_embeddings.shape)

Generating transcript embeddings... ⏳


Batches: 100%|██████████| 8/8 [00:16<00:00,  2.09s/it]

Transcript embeddings generated successfully ✅
Shape of transcript embeddings: (245, 384)


In [6]:
# -----------------------------
# Step 6: Combine Embeddings
# -----------------------------

print("Combining title and transcript embeddings... ⏳")

# Average the embeddings
combined_embeddings = (title_embeddings + transcript_embeddings) / 2

print("Combined embeddings created successfully ✅")

# Check shape
print("Shape of combined embeddings:", combined_embeddings.shape)

Combining title and transcript embeddings... ⏳
Combined embeddings created successfully ✅
Shape of combined embeddings: (245, 384)


In [7]:
# -----------------------------
# Step 7: Append Embeddings to Dataset
# -----------------------------

print("Appending embeddings to dataset... ⏳")

# Convert embeddings to DataFrame
embedding_df = pd.DataFrame(combined_embeddings)

# Rename columns → embedding_1, embedding_2, ...
embedding_df.columns = [f"embedding_{i}" for i in range(embedding_df.shape[1])]

# Combine with original dataset
final_df = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)

print("Embeddings appended successfully ✅")

# Check result
final_df.head()

Appending embeddings to dataset... ⏳
Embeddings appended successfully ✅


,video_id,title,datetime,transcript,embedding_0,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,...,embedding_374,embedding_375,embedding_376,embedding_377,embedding_378,embedding_379,embedding_380,embedding_381,embedding_382,embedding_383
0,ldkfMvE36FI,How to handle being thrown into an existing co...,2026-03-08 12:43:26+00:00,theres going to be a need for people can like ...,-0.017596,-0.095291,0.056069,-0.028207,0.039805,-0.015040,...,0.079234,0.013098,-0.023241,0.036684,-0.066737,0.071216,-0.078861,-0.008572,0.032594,0.015356
1,jie_039IekA,Why you shouldnt chase people to get what you ...,2026-03-07 13:32:30+00:00,I dont do a ton of followup Like you know if i...,-0.058301,0.013225,0.012424,-0.075241,-0.012175,-0.032885,...,0.013678,-0.054440,0.027800,0.021932,0.046389,0.025415,0.005570,-0.002358,-0.005379,0.029290
2,mrRfPVm9nAY,Learn the basics of Data Structures in 60 seco...,2026-03-06 13:18:50+00:00,Lets learn the basics of data structures in le...,-0.015724,0.019981,0.005629,-0.015846,-0.079441,-0.097639,...,0.022740,0.014300,0.000673,-0.036183,0.020679,0.021774,0.002178,0.013053,-0.016505,-0.004650
3,hP931079TMw,There are 2 kinds of devs One of them is screw...,2026-03-06 11:01:15+00:00,Welcome back to the Free Code Camp podcast Im ...,-0.056384,-0.045460,0.002062,-0.044062,-0.019443,-0.011673,...,0.056522,0.045513,0.004705,-0.057094,-0.025583,0.024735,-0.003302,0.036114,0.018302,-0.008661
4,tVskbekONlw,Learn MLOps with MLflow and Databricks Full Co...,2026-03-05 14:21:18+00:00,This course is an end toend guide to mastering...,-0.022512,-0.077987,0.026442,-0.015346,0.055578,-0.076017,...,0.089358,0.048688,0.019932,-0.056607,0.027377,0.053911,0.003975,0.035524,-0.008475,0.013456


In [8]:
# -----------------------------
# Step 8: Save Dataset (CSV)
# -----------------------------

output_path = "../data/video_index.csv"

final_df.to_csv(output_path, index=False)

print(f"Dataset saved as CSV at: {output_path} ✅")

Dataset saved as CSV at: ../data/video_index.csv ✅


In [10]:
# -----------------------------
# Step 9: Verify Embedding Data Integrity
# -----------------------------

print("Running validation checks... ⏳")

# 1️⃣ Check number of rows
print("\nTotal rows in dataset:", final_df.shape[0])
print("Total videos in original dataset:", df.shape[0])

if final_df.shape[0] == df.shape[0]:
    print("✅ Row count matches")
else:
    print("❌ Row count mismatch")

# 2️⃣ Check embedding columns
embedding_cols = [col for col in final_df.columns if "embedding_" in col]

print("\nTotal embedding columns:", len(embedding_cols))

# 3️⃣ Check for null values
null_count = final_df[embedding_cols].isnull().sum().sum()

print("Total null values in embeddings:", null_count)

if null_count == 0:
    print("✅ No missing embeddings")
else:
    print("❌ Missing values found")

# 4️⃣ Check embedding dimension consistency
embedding_length = len(embedding_cols)

print("\nEmbedding dimension:", embedding_length)

# Check one sample row
sample_embedding = final_df[embedding_cols].iloc[0].values

print("Sample embedding length:", len(sample_embedding))

if len(sample_embedding) == embedding_length:
    print("✅ Embedding dimensions consistent")
else:
    print("❌ Dimension mismatch")

print("\n Validation completed!")

Running validation checks... ⏳

Total rows in dataset: 245
Total videos in original dataset: 245
✅ Row count matches

Total embedding columns: 384
Total null values in embeddings: 0
✅ No missing embeddings

Embedding dimension: 384
Sample embedding length: 384
✅ Embedding dimensions consistent

 Validation completed!
